# 03 · Digital filters

Filters appear everywhere in a transceiver: anti-aliasing before a
decimation, the electrical bandwidth of a photodiode or a DAC, channel
selection, noise limiting.  CommKit keeps **design** and **application**
apart: a design function returns coefficients (`fir_taps`,
`butterworth_sos`, ...), and an apply function uses them (`fir_filter`,
`ols_fir_filter`, `iir_filter`).

1. FIR design by the window method: length against transition width;
2. applying an FIR filter, directly or by overlap-save;
3. the IIR families and their trade-off between selectivity and phase;
4. causal against zero-phase filtering;
5. what *smoothing* is for, and why it is not in the signal chain.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal as sps_signal

import commkit as ck
from commkit import Constellation
from commkit.backend import to_device

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()


def host(a):
    return to_device(a, "cpu")

## 1. FIR filters by the window method

An ideal low-pass filter has a sinc impulse response of infinite length.
The window method truncates it to `num_taps` samples and tapers the edges
with a window.  The result is **linear phase** (a symmetric impulse
response), so every frequency is delayed by the same `(num_taps - 1) / 2`
samples, and the transition band narrows as `1 / num_taps`.
`plot_filter_response` shows the impulse, magnitude, phase and group delay
of any design.

In [ ]:
fs = 1e6
taps = ck.filtering.fir_taps(sampling_rate=fs, num_taps=101, cutoff=100e3)
ck.plotting.plot_filter_response(taps, sampling_rate=fs, show=True)

fig, ax = plt.subplots(figsize=(8, 4))
for n in (21, 51, 201):
    w, h = sps_signal.freqz(
        host(ck.filtering.fir_taps(sampling_rate=fs, num_taps=n, cutoff=100e3)),
        worN=4096,
        fs=fs,
    )
    ax.plot(w / 1e3, 20 * np.log10(np.abs(h) + 1e-12), label=f"{n} taps")
ax.set_xlabel("Frequency [kHz]")
ax.set_ylabel("|H| [dB]")
ax.set_ylim(-90, 5)
ax.set_title("Hamming-window low-pass, cutoff 100 kHz")
ax.legend()
plt.show()

## 2. Applying a FIR filter

`fir_filter` convolves along the time axis and keeps the output aligned
with the input (the filter delay is removed), so sample `n` of the output
belongs to sample `n` of the input.  `ols_fir_filter` computes the same
convolution by overlap-save FFT blocks, which is much faster for long filters
(chromatic-dispersion compensation needs thousands of taps).

Here a 2 MBd QPSK signal sits next to a strong interferer at 3.5 MHz; a
low-pass filter removes it.

In [ ]:
tx = ck.generate(
    Constellation.psk(4),
    num_symbols=2**14,
    symbol_rate=2e6,
    sps=8,
    pulse=ck.RRC(rolloff=0.25),
    rng=1,
    device=DEVICE,
)
x = ck.spectral.add_pilot_tone(tx, frequency=3.5e6, power_ratio_db=10)

lp = ck.filtering.fir_taps(sampling_rate=tx.sampling_rate, num_taps=129, cutoff=1.6e6)
y_direct = ck.filtering.fir_filter(x, lp)
y_ols = ck.filtering.ols_fir_filter(x, lp)
err = np.max(np.abs(host(y_direct.samples) - host(y_ols.samples)))
print(f"direct vs overlap-save: max difference {err:.1e}")

fig, ax = plt.subplots(figsize=(8, 4))
ck.plotting.plot_psd(x, nperseg=1024, ax=ax, title=None, label="with interferer")
ck.plotting.plot_psd(
    y_direct, nperseg=1024, ax=ax, title="129-tap low-pass at 1.6 MHz", label="filtered"
)
ax.legend()
plt.show()

## 3. IIR families

An IIR filter reaches a given selectivity with far fewer coefficients, at
the cost of a nonlinear phase.  CommKit designs them as cascades of
second-order sections (SOS, numerically robust).  At the same order the
classic families trade steepness against phase linearity:

| design | passband | stopband | group delay |
|---|---|---|---|
| `bessel_sos` | smooth, slow roll-off | slow | flattest (best pulse fidelity) |
| `butterworth_sos` | maximally flat | moderate | moderate peak at the edge |
| `chebyshev1_sos` | ripple | steep | larger peak |
| `chebyshev2_sos` | flat | ripple, steep | moderate |
| `elliptic_sos` | ripple | ripple, steepest | largest peak |

A Bessel response is the usual model of a receiver's electrical bandwidth
because its flat group delay keeps pulses undistorted.

In [ ]:
designs = {
    "Bessel": ck.filtering.bessel_sos(sampling_rate=fs, cutoff=100e3, order=6),
    "Butterworth": ck.filtering.butterworth_sos(
        sampling_rate=fs, cutoff=100e3, order=6
    ),
    "Chebyshev I": ck.filtering.chebyshev1_sos(
        sampling_rate=fs, cutoff=100e3, order=6, ripple=1.0
    ),
    "Chebyshev II": ck.filtering.chebyshev2_sos(
        sampling_rate=fs, cutoff=100e3, order=6, attenuation=50.0
    ),
    "Elliptic": ck.filtering.elliptic_sos(
        sampling_rate=fs, cutoff=100e3, order=6, ripple=1.0, attenuation=50.0
    ),
}
fig, (ax_m, ax_g) = plt.subplots(1, 2, figsize=(12, 4))
passband = np.linspace(0, 100e3, 400)
for name, sos in designs.items():
    sos = host(sos)
    w, h = sps_signal.sosfreqz(sos, worN=4096, fs=fs)
    ax_m.plot(w / 1e3, 20 * np.log10(np.abs(h) + 1e-12), label=name)
    # The group delay of a cascade is the sum over its sections.
    gd = sum(
        sps_signal.group_delay((sec[:3], sec[3:]), w=passband, fs=fs)[1] for sec in sos
    )
    ax_g.plot(passband / 1e3, gd / fs * 1e6, label=name)
ax_m.set_ylim(-80, 5)
ax_m.set_xlabel("Frequency [kHz]")
ax_m.set_ylabel("|H| [dB]")
ax_m.set_title("Order 6, cutoff 100 kHz")
ax_g.set_xlabel("Frequency [kHz]")
ax_g.set_ylabel("Group delay [µs]")
ax_g.set_title("Group delay across the passband")
ax_m.legend()
plt.show()

The group-delay peak at the band edge shows up in the time domain as
overshoot and ringing.  The steep designs ring for many samples; the Bessel
step barely overshoots.  (Even-order Chebyshev I and elliptic designs also
settle at the bottom of their 1 dB passband ripple, a gain of 0.89 at DC.)

In [ ]:
step = np.r_[np.zeros(50), np.ones(200)].astype(np.float32)
fig, ax = plt.subplots(figsize=(8, 4))
for name, sos in designs.items():
    y = host(ck.filtering.iir_filter(step, sos, zero_phase=False))
    print(
        f"{name:13s} overshoot {100 * (y.max() / y[-1] - 1):5.1f} %, final value {y[-1]:.3f}"
    )
    ax.plot(np.arange(step.size) / fs * 1e6, y, label=name)
ax.set_xlabel("Time [µs]")
ax.set_ylabel("Step response")
ax.legend()
plt.show()

## 4. Causal or zero-phase

`iir_filter(..., zero_phase=True)` (the default) filters forward and then
backward: the phase cancels, the magnitude response is squared, and there
is no delay.  This needs the whole record and models no real device, but
it is what offline analysis wants.  `zero_phase=False` is the causal filter
a real-time device implements, with its frequency-dependent delay.

In [ ]:
pulse = np.zeros(300, dtype=np.float32)
pulse[100:110] = 1.0
sos = designs["Butterworth"]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(pulse, label="input")
ax.plot(
    host(ck.filtering.iir_filter(pulse, sos, zero_phase=False)),
    label="causal (delayed)",
)
ax.plot(
    host(ck.filtering.iir_filter(pulse, sos, zero_phase=True)),
    label="zero-phase (centered)",
)
ax.set_xlim(80, 160)
ax.set_xlabel("Sample")
ax.legend()
plt.show()

## 5. Smoothing is for looking, not for the signal chain

`commkit.smoothing` (`moving_average`, `savgol_smooth`,
`smooth_density_2d`) smooths *derived* quantities for display or
estimation: an error curve, a phase trajectory, a histogram.  They have no
meaningful frequency response in a link budget, so they never sit in the
signal path; use the filters above for that.

In [ ]:
rng = np.random.default_rng(2)
curve = np.exp(-np.arange(2000) / 400) + 0.05 * rng.standard_normal(2000)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(curve, alpha=0.4, label="raw (e.g. an equalizer's squared error)")
ax.plot(ck.smoothing.moving_average(curve, window=51), label="moving average, 51")
ax.plot(
    ck.smoothing.savgol_smooth(curve, window=101, polyorder=3),
    label="Savitzky-Golay, 101 / 3",
)
ax.legend()
plt.show()

**Summary**

* Design once, apply anywhere: `fir_taps` / `*_sos` return coefficients;
  `fir_filter`, `ols_fir_filter` and `iir_filter` apply them on the data's
  device.
* FIR: linear phase, transition width `∝ 1/num_taps`; overlap-save for long
  filters.
* IIR: selectivity against group-delay flatness; Bessel for pulse fidelity.
* `zero_phase=True` for offline analysis, `False` to model a real device.

**Next:** [04 · Pulse shaping and matched filtering](04-pulse-shaping-and-matched-filtering.ipynb).